## ការ​ដាក់សញ្ញា Embeddings

ក្នុងឧទាហរណ៍មុនរបស់យើង យើងបានដំណើរការលើវ៉ិចទ័រចំនួនពាក្យដែលមានការរៀបចំជារង្វង់លំដាប់ខ្ពស់មានប្រវែង `vocab_size` ហើយយើងកំពុងបម្លែងយ៉ាងច្បាស់ពីវ៉ិចទ័រតំណាងលំដាប់ទាបទៅសារពត៌មាន one-hot ដែលចង្អុលបង្ហាញមិនយ៉ាងដូចគ្នា។ ការ​តំណាង one-hot នេះ​មិនជា​ភាព​ប្រសើរ​ទាំងមូល​នៃ​អង្គចងចាំទេ លើសពីនេះមកមួយពាក្យត្រូវបានយកទៅគិតដោយឡែកពីពាក្យផ្សេងទៀត មានន័យថាវ៉ិចទ័រ one-hot មិនបង្ហាញពីស្រមោលនៃអត្ថន័យត្រូវគ្នារវាងពាក្យនីមួយៗទេ។

ក្នុងឯកតានេះ យើងនឹងបន្តស្វែងយល់ពាក់ព័ន្ធនឹង **News AG** សំណុំទិន្នន័យ។ ដើម្បីចាប់ផ្តើម សូមយើងផ្ទុកទិន្នន័យនិងទទួលយកកំណត់ប្រភេទខ្លះៗពីសៀវភៅកំណត់ត្រាមុន។


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## embedding គឺជាអ្វី?

គំនិតនៃ **embedding** គឺតំណាងឲ្យពាក្យជាវិចទ័រមានមิติទាបដែលមានភាពរាប់រយខ្ពស់ ដែលជារបៀបមួយបង្ហាញន័យសមាសភាពនៃពាក្យមួយ។ យើងនឹងពិភាក្សាពីរបៀបបង្កើត embedding ពាក្យមានន័យនៅក្រោយនេះ ប៉ុន្តែសម្រាប់ពេលនេះ សូមគ្រាន់តែគិតថា embedding ជារបៀបមួយបង្រួមមิติវិចទ័ររបស់ពាក្យមួយ។ 

ដូច្នេះ, ស្រទាប់ embedding នឹងទទួលពាក្យមួយជាឧបញ្ញាណ និងបង្កើតវិចទ័រចេញមួយដែលមានមិត្តិដែលកំណត់ `embedding_size`។ ជារបៀបមួយ វាស្រដៀងគ្នាជាមណ៌ម៉ង `Linear` ប៉ុន្តែជំនួសយកវិចទ័រតើបរើក​មួយ ស្រទាប់ embedding អាចទទួលលេខពាក្យមួយជាឧបញ្ញាណបាន។

ដោយប្រើស្រទាប់ embedding ជាស្រទាប់ដំបូងក្នុងបណ្តាញយើង អាចផ្លាស់ប្តូរពីបណ្ណាល័យពាក្យទៅគំរូ **embedding bag** ដែលយើងដំណើរការបម្លែងពាក្យនីមួយៗនៅក្នុងអត្ថបទទៅ embedding តាមលំដាប់ ហើយបន្ទាប់មកគណនាអនុគមន៍សរុបមួយលើបញ្ចូល embedding ទាំងអស់ ដូចជា `sum`, `average` ឬ `max`។  

![រូបភាពបង្ហាញពីម៉ូដែលគណនាបន្ទាន់មួយសម្រាប់ពាក្យខ្សែភាពបួន.](../../../../../translated_images/km/embedding-classifier-example.b77f021a7ee67eee.webp)

បណ្តាញសរសេរច្បាប់របស់យើងនឹងចាប់ផ្តើមជាមួយស្រទាប់ embedding បន្ទាប់ស្រទាប់ស្ដាប់សរុប ហើយបន្ទាប់មកជាស្រទាប់ក្ដារលីនេអ៊ែរ៖


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### ការដំណោះស្រាយទំហំលំដាប់អថេរ

ជាលទ្ធផលនៃស្ថាបត្យកម្មនេះ minibatches ទៅកាន់បណ្ដាញរបស់យើងត្រូវតែត្រូវបានបង្កើតឡើងក្នុងវិធីមួយជាក់លាក់។ ក្នុងអង្គភាពមុន បើពេលប្រើ bag-of-words យើងបានឃើញថា ក្នុង minibatch ទាំងអស់ BoW tensors មានទំហំស្មើគ្នា `vocab_size` មិនគិតពីប្រវែងពិតប្រាកដនៃលំដាប់អត្ថបទរបស់យើងឡើយ។ ពេលដែលយើងផ្លាស់ប្តូរទៅជាការតំណាងពាក្យ word embeddings យើងនឹងមានចំនួនពាក្យប្រែប្រួលក្នុងគំរូអត្ថបទនីមួយៗ ហើយពេលបញ្ចូលគំរូទាំងនេះទៅក្នុង minibatches យើងត្រូវតែអនុវត្ត padding។

របៀបនេះអាចធ្វើបានដោយប្រើបច្ចេកទេសដូចគ្នានៃការផ្តល់មុខងារ `collate_fn` ទៅកាន់ប្រភពទិន្នន័យៈ


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### បណ្តុះបណ្តាលអ្នកចាត់ថ្នាក់ embedding

ឥឡូវនេះ យើងបានកំណត់ dataloader យ៉ាងត្រឹមត្រូវ ហើយ យើងអាចបង្រៀនម៉ូដែលដោយប្រើមុខងារបណ្ដុះបណ្ដាល ដែលយើងបានកំណត់នៅឯកម្សៅមុន៖


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **កំណត់ចំណាំ**៖ យើងកំពុងបង្រៀនសម្រាប់កំណត់ត្រា 25k នៅទីនេះតែប៉ុណ្ណោះ (តិចជាងមួយវដ្តពេញ) ដើម្បីប្រើពេលវេលា ប៉ុន្តែអ្នកអាចបន្តបង្រៀន បិទសរសេរមុខងារមួយដើម្បីបង្រៀនរយៈពេលជាច្រើនវដ្ត ហើយសាកល្បងជាមួយប៉ារម៉ែត្រ​អត្រាសិក្សា ដើម្បីទទួលបានភាពត្រឹមត្រូវកម្រិតខ្ពស់។ អ្នកគួរតែអាចទៅដល់ភាពត្រឹមត្រូវប្រហែល ៩០%។  


### ស្រទាប់ EmbeddingBag និងការតំណាងស៊េរីមានប្រវែងប្រែប្រួល

ក្នុងស្ថាបត្យកម្មមុន, យើងបានត្រូវការដាក់ពង្រីកស៊េរីទាំងអស់ឲ្យមានប្រវែងដូចគ្នាដើម្បីអាចដាក់វាចូលក្នុងមីនីបាច់មួយ។ វាមិនមែនជាវិធីយុទ្ធសាស្រ្តល្អបំផុតក្នុងការតំណាងស៊េរីដែលមានប្រវែងប្រែប្រួលទេ - វិធីផ្សេងគឺការប្រើវ៉ិចទ័រ **offset** ដែលជាកន្លែងរក្សាទុក offset របស់ស៊េរីទាំងអស់ដែលបានរក្សាទុកក្នុងវ៉ិចទ័រធំមួយ។

![រូបភាពបង្ហាញការតំណាងស៊េរី offset](../../../../../translated_images/km/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **ចំណាំ**៖ នៅលើរូបភាពខាងលើ យើងបង្ហាញស៊េរីតួអក្សរមួយ ប៉ុន្តែក្នុងឧទាហរណ៍របស់យើង យើងកំពុងប្រើស៊េរីពាក្យ។ ទោះយ៉ាងណា គោលការណ៍ទូទៅនៃការតំណាងស៊េរីជាមួយវ៉ិចទ័រ offset នៅតែដូចគ្នា។

ដើម្បីធ្វើការ​ជាមួយការតំណាង offset, យើងប្រើស្រទាប់ [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html)។ វាស្រដៀងនឹង `Embedding` តែវាចូលទទួលវ៉ិចទ័រមាតិកាប្រភេទ content និងវ៉ិចទ័រ offset ជាទិន្នន័យចូល ហើយវាក៏រួមបញ្ចូលស្រទាប់ averaging ដែលអាចជាទាំង `mean`, `sum` ឬ `max` ផងដែរ។

នេះជាបណ្ដាញដែលបានកែប្រែដែលប្រើប្រាស់ `EmbeddingBag`៖


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

ដើម្បីរៀបចំមូលដ្ឋានទិន្នន័យសម្រាប់ការបណ្តុះបណ្តាល យើងត្រូវតែផ្តល់មុខងារបម្លែងមួយដែលនឹងរៀបចំវ៉ិចទ័រយកលម្អិត៖


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

កំណត់សម្គាល់ ថា ខុសពីគំរូមុនៗទាំងអស់ បណ្តាញរបស់យើងឥឡូវនេះទទួលបានពារ៉ាម៉ែត្រពីរ៖ វ៉ិចទ័រទិន្នន័យ និងវ៉ិចទ័របង្រួម ដែលមានទំហំខុសគ្នា។ ដូចគ្នា អ្នកផ្ទុកទិន្នន័យរបស់យើងក៏ផ្តល់ជូនពណ៌តម្លៃ ៣ ផ្ទុយពី ២ ផងដែរ៖ វ៉ិចទ័រអត្ថបទ និងវ៉ិចទ័របង្រួមទាំងពីរត្រូវបានផ្តល់ជាលក្ខណៈលក្ខណៈពិសេស។ ដូច្នេះ យើងត្រូវតែថែរក្សាអនុគមន៍បណ្តុះបណ្តាលរបស់យើងបន្តិចបន្តួចដើម្បីកែលម្អនេះ៖


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## ការចាក់បញ្ចូលអត្ថន័យ៖ Word2Vec

ក្នុងឧទាហរណ៍មុនរបស់យើង ស្រទាប់ចាក់បញ្ចូលម៉ូដែលបានរៀនបំលែងពាក្យទៅតំណាងវ៉ិចទ័រមួយ ប៉ុន្តែតំណាងនេះមិនមានន័យអត្ថន័យច្រើនទេ។ វានឹងល្អបើរៀនបានតំណាងវ៉ិចទ័រដូច្នេះ ដែលពាក្យស្រដៀងគ្នា ឬពាក្យប្រើប្រាស់ដូចគ្នា នឹងត្រូវតំណាងដោយវ៉ិចទ័រដែលនៅជិតគ្នាទៅតាមចម្ងាយវ៉ិចទ័រមួយចំនួន (ឧ. ចម្ងាយអ៊ូលស៊ីឌាំង)។

ដើម្បីធ្វើវា យើងត្រូវតែមួយរៀនម៉ូដែលចាក់បញ្ចូលរបស់យើងលើការបណ្ដុះបណ្ដាលអត្ថបទធំមួយយ៉ាងពិសេស។ មួយក្នុងចំណោមវិធីដំបូងនៃការបណ្តុះបណ្តាលអត្ថន័យចាក់បញ្ចូលគឺហៅថា [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)។ វាអាស្រ័យលើស្ថាបត្យាកម្មសំខាន់ពីរដែលប្រើសម្រាប់ផលិតតំណាងពាក្យចែកចាយ៖

 - **ថង់ពាក្យបន្តបឋម** (CBoW) — ក្នុងស្ថាបត្យកម្មនេះ យើងបណ្តុះម៉ូដែលឲ្យព្យាករ​ពាក្យពីសេចក្ដីបរិវេណជុំវិញ។ បើបាន ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$ គោលដៅរបស់ម៉ូដែលគឺព្យាករ $W_0$ ពី $(W_{-2},W_{-1},W_1,W_2)$។
 - **ប្រើប្រាស់ skip-gram បន្តបឋម** គឺផ្ទុយទៅវិញនឹង CBoW. ម៉ូដែលប្រើបរិវេណពាក្យជុំវិញដើម្បីព្យាករពាក្យបច្ចុប្បន្ន។

CBoW ឆាប់រហ័ស ក្រោយមក skip-gram យឺតជាង ប៉ុន្តែធ្វើការល្អជាងសម្រាប់តំណាងពាក្យដែលកើតឡើងតិច។

![រូបភាពបង្ហាញអាល់ហ្គរីធម៍ទាំង CBoW និង Skip-Gram សម្រាប់បម្លែងពាក្យទៅវ៉ិចទ័រ។](../../../../../translated_images/km/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

ដើម្បីសាកល្បងជាមួយការចាក់បញ្ចូល word2vec ដែលបានបណ្តុះបណ្តាលជាមុនលើសំណុំទិន្នន័យ Google News យើងអាចប្រើបណ្ណាល័យ **gensim**។ ខាងក្រោម យើងស្វែងរកពាក្យដែលស្រដៀងបំផុតនឹង 'neural'

> **ចំណាំ៖** ពេលអ្នកបង្កើតវ៉ិចទ័រពាក្យជាលើកដំបូង ការទាញយកវាអាចចំណាយពេលខ្លះ!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


យើងក៏អាចគណនាការបញ្ចាំងវ៉ិចទ័រពីពាក្យ ដើម្បីប្រើប្រាស់ក្នុងការបង្រៀនម៉ូដែលចាត់ថ្នាក់ (យើងបង្ហាញតែក្រឡាផ្នែកទី ២០ ដំបូងនៃវ៉ិចទ័រដើម្បីភាពច្បាស់): 


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

វិសេសភាពដ៏អស្ចារ្យអំពីការរួមបញ្ចូលន័យ គឺអ្នកអាចគ្រប់គ្រងការអ៊ិនកូដវ៉ិចទ័រដើម្បីផ្លាស់ប្តូរន័យនៃពាក្យ។ ឧទាហរណ៍ យើងអាចស្នើសុំស្វែងរកពាក្យមួយ ដែលតំណាងវ៉ិចទ័ររបស់វានឹងជិតបំផុតទៅពាក្យ *king* និង *woman* ហើយឆ្ងាយពីពាក្យ *man*៖


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

ទាំង CBoW និង Skip-Grams គឺ​ជា​ការបង្ហាញ “ទំនោរ” ដែល​ត្រឹមតែ​គិតពី​បរិបទ​ក្នុង​តំបន់​តែប៉ុណ្ណោះ។ Word2Vec មិន​ទទួល​បាន​អត្ថប្រយោជន៍​ពី​បរិបទ​សកល​ឡើយ។  

**FastText** សាងសង់​លើ Word2Vec ដោយ​រៀន​តំណាង​វិចិត្រសម្រាប់​ពាក្យ​រាល់ពាក្យ និង​និម្មិត​អក្សរ n-gram ដែល​រកឃើញ​ក្នុង​ពាក្យ​នីមួយៗ។ តម្លៃ​នៃ​តំណាង​នេះ​បន្ទាប់មក​ត្រូវ​បាន​សង្វាក់ទៅជាវិចិត្រដូចមួយ​នៅ​ក្នុង​ជំហាន​បណ្តុះបណ្តាល​នីមួយៗ។ ខណៈដែល​នេះ​បន្ថែម​ការ​គណនា​ច្រើន​ក្នុង​ការ​បណ្តុះបណ្តាល​មុន វាជួយ​ឲ្យ​ការ​បង្ហាញ​ពាក្យ​អាច​លេខាអក្សរ​តូចៗ​ក្នុង​ពាក្យ​បាន។  

វិធីមួយទៀត​គឺ **GloVe** ដែល​យក​គំនិត​នៃ matrix ជួប​គ្នា (co-occurrence matrix) ប្រើវិធីសាស្ដ្រ​នៃ​សំណួរ​ជាលេខ​ដើម្បី​បំបែក co-occurrence matrix ទៅ​ជា​វិចិត្រពាក្យ​ដែល​មាន​ការ​បង្ហាញ​ប្រសើរ និង​មិន​រាយលំដាប់ (non linear)។  

អ្នកអាចលេងជាមួយឧទាហរណ៍ដោយបម្លែង embeddings ទៅជា FastText និង GloVe ពីព្រោះ gensim គាំទ្រគំរូការបង្ហាញពាក្យជាច្រើន។  


## ការប្រើប្រាស់អាំងប៊ែឌីងដែលបានបណ្ដុះបណ្ដាលជាមុននៅក្នុង PyTorch

យើងអាចផ្លាស់ប្តូរឧទាហរណ៍ខាងលើដើម្បីបញ្ចូល Matrix នៅក្នុងស្រទាប់អាំងប៊ែឌីងរបស់យើងជាមុនជាមួយអាំងប៊ែឌីងមានន័យដូចជា Word2Vec។ យើងត្រូវបានគិតគូរថា វាក្យសព្ទនៃអាំងប៊ែឌីងដែលបានបណ្ដុះបណ្ដាលជាមុន និងអត្ថបទក្នុងអំពើពួកយើងប្រហែលជាមិនត្រូវគ្នាទេ ដូច្នេះយើងនឹងផ្តើមតម្លៃរ៉ាប់រ៉ៃសម្រាប់ពាក្យដែលខ្វះនៅក្នុងពួកវា៖


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


ឥឡូវនេះយើងមកហ្វឹកហាត់ម៉ូដែលរបស់យើង។ ចំណាំថាពេលវេលាដែលត្រូវប្រើសម្រាប់ហ្វឹកហាត់ម៉ូដែលគឺយ៉ាងសំខាន់ធំជាងក្នុងឧទាហរណ៍មុននេះ ដោយសារតែទំហំស្រទាប់បញ្ចូលដែលធំនិងចំនួនប៉ារ៉ាម៉ែត្រ ដែលខ្ពស់ជាច្រើន។ ក៏ដោយហេតុនេះផងដែរ យើងប្រហែលជាត្រូវហ្វឹកហាត់ម៉ូដែលរបស់យើងលើឧទាហរណ៍ច្រើនជាងនេះ ប្រសិនបើយើងចង់ជៀសវាងការហ្វឹកហាត់លើតំរូវការ។


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

ក្នុងករណីរបស់យើង យើងមិនឃើញការកើនឡើងយ៉ាងច្រើននៅក្នុងភាពត្រឹមត្រូវទេ ដែលប្រហែលជាមកពីវាក្យសព្ទខុសគ្នាយ៉ាងខ្លាំង។ 
ដើម្បីដោះស្រាយបញ្ហាវាក្យសព្ទខុសគ្នា យើងអាចប្រើមធ្យោបាយមួយក្នុងចំណោមដូចខាងក្រោម៖
* ហ្វឹកហាត់ម៉ូដែល word2vec ម្តងទៀតលើវាក្យសព្ទរបស់យើង
* ផ្ទុកឯកសារទិន្នន័យរបស់យើងជាមួយវាក្យសព្ទពីម៉ូដែល word2vec ដែលបានហ្វឹកហាត់រួចហើយ។ វាក្យសព្ទដែលបានប្រើសម្រាប់ផ្ទុកឯកសារទិន្នន័យអាចកំណត់បាននៅពេលផ្ទុក។

វិធីសាស្រ្តចុងក្រោយនេះហាក់ដូចជាងាយស្រួលជាងគេ ពិសេសដោយសារតែស៊ុមគ្លីប PyTorch `torchtext` មានការគាំទ្រក្នុងការបញ្ចូលដែលបានបញ្ចូលរួចហើយ។ យើងអាច ឧទាហរណ៍ បង្កើតវាក្យសព្ទផ្អែកលើ GloVe ដូចខាងក្រោម៖


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


វាក្យាករណ៍ដែលបានផ្ទុកមានប្រតិបត្តិការមូលដ្ឋានដូចខាងក្រោម៖
* `vocab.stoi` មនុស្សអាចបម្លែងពាក្យទៅជាអនុគមន៍តារាងវចនានុក្រមរបស់វា
* `vocab.itos` ធ្វើការផ្ទុយ - បម្លែងលេខទៅជាពាក្យ
* `vocab.vectors` គឺជា​អារេ​នៃវ៉ិកទ័រ embedding ដើម្បីទទួលបាន embedding របស់ពាក្យ `s` យើងត្រូវប្រើ `vocab.vectors[vocab.stoi[s]]`

នេះគឺជាឧទាហរណ៍នៃការបញ្ជូនការបង្ហាញ (embeddings) ដើម្បីបង្ហាញសមីការ **kind-man+woman = queen** (ខ្ញុំត្រូវបានបត់បែនអត្រាដើម្បីឲ្យវាដំណើរការ):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ដើម្បីហ្វឹកហាត់អ្នកចាត់ថ្នាក់ដោយប្រើការចាក់បញ្ចូលទាំងនោះ យើងត្រូវការកូដកំណត់ឯកសារយោងតាមវចនានុក្រម GloVe មុនសិន៖


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

ដូចដែលយើងបានឃើញខាងលើ embedding vectors ទាំងអស់ ត្រូវបានរក្សាទុកក្នុងមាទ្រីស `vocab.vectors` ។ វាធ្វើឱ្យមានភាពងាយស្រួលយ៉ាងខ្លាំងក្នុងការលិចបញ្ចូលទម្ងន់ទាំងនោះទៅក្នុងអម្ពិលទម្ងន់របស់ស្រទាប់ embedding ដោយប្រើការចម្លងបានយ៉ាងសាមញ្ញ៖


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

ឥឡូវនេះយើងចូរបណ្តុះគំរូរបស់យើង ហើយមើលថាតើយើងទទួលបានលទ្ធផលល្អជាងមុនឬអត់៖


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

មូលហេតុមួយដែលធ្វើឱ្យយើងមិនឃើញការកើនឡើងយ៉ាងសំខាន់ក្នុងភាពត្រឹមត្រូវគឺដោយសារពាក្យខ្លះៗពីបណ្ណាល័យទិន្នន័យរបស់យើងបាត់នៅក្នុងវាក្យសព្ទដែលបានបណ្ដុះបណ្ដាល GloVe ដូច្នេះពួកវាត្រូវបានមើលរំលងជាសំខាន់។ ដើម្បីលើកកម្ពស់ចំណុចនេះ យើងអាចបណ្ដុះបណ្ដាល embeddings របស់យើងនៅលើបណ្ណាល័យទិន្នន័យរបស់យើង។ 


##​ បទបញ្ញត្តិក្នុងបរិបទ

មួយក្នុងចំណោមកំណត់ខ្លះៗរបស់តំណាងបញ្ចូលមុនផលិតដែលប្រពៃណីដូចជា Word2Vec គឺបញ្ហាការបំភ្លឺន័យពាក្យ។ ខណៈដែលតំណាងបញ្ចូលមុនផលិតអាចចាប់យកអត្ថន័យមួយចំនួននៃពាក្យក្នុងបរិបទ តែន័យសកលមួយៗនៃពាក្យត្រូវបានគណនាចូលក្នុងតំណាងបញ្ចូលដូចគ្នា។ នេះអាចបណ្តាលឲ្យមានបញ្ហានៅក្នុងគំរូក្រោម បើទោះយ៉ាងណា ពាក្យជាច្រើន ដូចជា​ពាក្យ 'play' មានន័យខុសៗគ្នាដាច់ពីគ្នា ដោយសារបរិបទដែលវាត្រូវបានប្រើ។

ឧទាហរណ៍ ពាក្យ 'play' ក្នុងប្រយោគទាំងពីរនេះមានន័យខុសគ្នា:
- ខ្ញុំបានទៅឃើញ **ការលេងល្បែង** នៅក្នុងរដ្ឋាភិបាល។
- John ចង់ **លេង** ជាមួយមិត្តភក្តិរបស់គាត់។

តំណាងបញ្ចូលមុនផលិតខាងលើបង្ហាញទាំងពីរន័យនៃពាក្យ 'play' នៅក្នុងតំណាងបញ្ចូលដូចគ្នា។ ដើម្បីអាចទប់ស្កាត់កំណត់នេះ យើងត្រូវតែបង្កើតតំណាងបញ្ចូលជាប់លាប់លើ **គំរូភាសា** ដែលបានបណ្ដុះបណ្ដាលលើសរុបទិន្នន័យអក្សរច្រើន ហើយ *ដឹង* របៀបដែលពាក្យអាចត្រូវបានដាក់រួមគ្នានៅក្នុងបរិបទផ្សេងៗគ្នា។ ការពិភាក្សារអំពីតំណាងបញ្ចូលដើម្បីបរិបទគឺមិននៅក្នុងវិស័យសិក្សារបស់មេរៀននេះទេ ប៉ុន្តែយើងនឹងត្រឡប់មកវិញពេលពិភាក្សាអំពីគំរូភាសានៅ និច្ចខាងក្រោម។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
